# Phase 0 — Upstash 적재

> **1회만 실행한다.** 이후 모든 노트북(phase2~7)은 이 노트북이 적재한 데이터를 검색만 한다.

### 적재 대상
| 스토어 | 데이터 | 네임스페이스 | 건수 |
|--------|--------|-------------|------|
| DualUpstashStore | SAMPLE_ENGINEER_PROFILES | capability / experience | 100명 |
| CapabilityMasterStore | SAMPLE_SKILLS | capability_master | 84개 |

### idempotent guard
- 이미 적재된 상태면 SKIP (재실행해도 중복 적재 안 됨)
- 강제 재적재가 필요하면 `skip_if_exists=False` 로 변경

In [11]:
from _bootstrap import setup_project_path
setup_project_path()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [12]:
import os
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores import DualUpstashStore, CapabilityMasterStore
from embedding_retrieval.services import ingest_profiles, ingest_skills
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES, SAMPLE_SKILLS

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)

url   = config.vector_store_kwargs["url"]
token = config.vector_store_kwargs["token"]

print(f"임베딩 모델  : {config.embedding_provider} / {config.embedding_model}")
print(f"Upstash URL : {url[:40]}..." if len(url) > 40 else f"Upstash URL : {url}")
print(f"프로필 수    : {len(SAMPLE_ENGINEER_PROFILES)}명")
print(f"스킬 수      : {len(SAMPLE_SKILLS)}개")

임베딩 모델  : google_genai / gemini-embedding-2-preview
Upstash URL : https://rare-rat-65880-us1-vector.upstas...
프로필 수    : 100명
스킬 수      : 84개


## Cell 1 — 환경 확인

Upstash 연결 및 스토어 초기화 상태를 확인한다.

In [13]:
# 스토어 초기화
dual_store   = DualUpstashStore(embeddings=embeddings, url=url, token=token)
master_store = CapabilityMasterStore(embeddings=embeddings, url=url, token=token)

# 연결 확인 — is_empty() 가 에러 없이 실행되면 연결 성공
try:
    skills_empty    = master_store.is_empty()
    print("[OK] Upstash 연결 성공")
    print(f"  capability_master 비어 있음: {skills_empty}")
except Exception as e:
    print(f"[FAIL] Upstash 연결 실패: {e}")
    raise

[OK] Upstash 연결 성공
  capability_master 비어 있음: True


## Cell 2 — 이미 적재됐는지 확인 (idempotent guard)

- 이미 데이터가 있으면 재적재 없이 SKIP
- 아직 없으면 Cell 3, 4 에서 적재

In [14]:
from embedding_retrieval.services.profile_ingest import _has_data

profiles_exist = _has_data(dual_store)
skills_exist   = not master_store.is_empty()

print("=== 적재 현황 ===")
print(f"  EngineerProfile (DualUpstashStore) : {'이미 적재됨 → SKIP' if profiles_exist else '미적재 → 적재 필요'}")
print(f"  Skill (CapabilityMasterStore)      : {'이미 적재됨 → SKIP' if skills_exist   else '미적재 → 적재 필요'}")

=== 적재 현황 ===
  EngineerProfile (DualUpstashStore) : 미적재 → 적재 필요
  Skill (CapabilityMasterStore)      : 미적재 → 적재 필요


## Cell 2-R — Upstash 리셋 (재적재 필요 시만 실행)

> ⚠️ **기존 데이터를 완전히 삭제한다.** 메타데이터 구조 변경 후 재적재할 때만 실행.  
> 정상 운영 중에는 이 셀을 실행하지 않는다.

In [15]:
# ⚠️ 기존 데이터 초기화 — 메타데이터 구조 변경 후 재적재 시에만 실행
# 실행하면 capability / experience / capability_master 전체 삭제됨
from upstash_vector import Index

index = Index(url=url, token=token)
for ns in ["capability", "experience", "capability_master"]:
    index.reset(namespace=ns)
    print(f"  {ns} 리셋 완료")

print("\n리셋 완료 — Cell 3, 4 를 다시 실행하여 재적재하세요.")

  capability 리셋 완료
  experience 리셋 완료
  capability_master 리셋 완료

리셋 완료 — Cell 3, 4 를 다시 실행하여 재적재하세요.


## Cell 3 — EngineerProfile 100개 적재

- capability_text → `capability` 네임스페이스
- experience_text → `experience` 네임스페이스
- 이미 적재된 경우 SKIP (0 반환)

In [16]:
ingested_profiles = ingest_profiles(
    profiles=SAMPLE_ENGINEER_PROFILES,
    store=dual_store,
    skip_if_exists=True,
)

if ingested_profiles == 0:
    print("[SKIP] EngineerProfile 이미 적재됨")
else:
    print(f"[OK]   EngineerProfile {ingested_profiles}명 적재 완료")
    print("       capability 네임스페이스: capability_text 임베딩")
    print("       experience 네임스페이스: experience_text 임베딩")

  적재 20/100 완료
  적재 40/100 완료
  적재 60/100 완료
  적재 80/100 완료
  적재 100/100 완료
  적재 20/100 완료
  적재 40/100 완료
  적재 60/100 완료
  적재 80/100 완료
  적재 100/100 완료
[OK]   EngineerProfile 100명 적재 완료
       capability 네임스페이스: capability_text 임베딩
       experience 네임스페이스: experience_text 임베딩


## Cell 4 — Skill 84개 적재

- embed_text(=name) → `capability_master` 네임스페이스
- 이미 적재된 경우 SKIP (0 반환)

In [17]:
ingested_skills = ingest_skills(
    skills=SAMPLE_SKILLS,
    store=master_store,
    skip_if_exists=True,
)

if ingested_skills == 0:
    print("[SKIP] Skill 이미 적재됨")
else:
    print(f"[OK]   Skill {ingested_skills}개 적재 완료")
    print("       capability_master 네임스페이스: embed_text(=name) 임베딩")

  적재 20/84 완료
  적재 40/84 완료
  적재 60/84 완료
  적재 80/84 완료
  적재 84/84 완료
[OK]   Skill 84개 적재 완료
       capability_master 네임스페이스: embed_text(=name) 임베딩


## Cell 5 — 적재 결과 검증

실제로 데이터가 저장됐는지 검색으로 확인한다.

In [23]:
# 5-1. capability_master 검증 — "Java" 검색 top-3
print("=== capability_master 검색: 'Java' top-3 ===")
results = master_store.search("java", top_k=3)
for r in results:
    print(f"  name={r.name:<20s} category={r.category:<12s} score={r.score:.4f}")

=== capability_master 검색: 'Java' top-3 ===
  name=Java                 category=LANGUAGE     score=0.9557
  name=JavaScript           category=LANGUAGE     score=0.8920
  name=C#                   category=LANGUAGE     score=0.8601


In [27]:
# 5-2. capability_master 검증 — "스프링부트" 검색 (한글 → 유사도 확인)
print("=== capability_master 검색: '스프링부트' top-3 ===")
results = master_store.search("Spring", top_k=3)
for r in results:
    print(f"  name={r.name:<20s} category={r.category:<12s} score={r.score:.4f}")

=== capability_master 검색: '스프링부트' top-3 ===
  name=Spring Boot          category=FRAMEWORK    score=0.8607
  name=Spring Security      category=FRAMEWORK    score=0.8523
  name=Spring Batch         category=FRAMEWORK    score=0.8290


In [33]:
# 5-3. DualUpstashStore 검증 — capability 검색
print("=== capability 검색: 'Java Spring Boot' top-3 ===")
cap_results = dual_store.search_capability("Java Spring Boot", top_k=3)
for r in cap_results:
    eid  = r.metadata.get("engineer_id", "?")
    role = r.metadata.get("engineer_role", "?")
    grade = r.metadata.get("grade", "?")
    capability_text = r.document.page_content   # ← metadata 아닌 document에서
    print(f"  engineer_id={eid:<10s} role={role:<12s} grade={grade:<14s} score={r.score:.4f}")
    print(f"    capability_text: {capability_text[:1000]}")

=== capability 검색: 'Java Spring Boot' top-3 ===
  engineer_id=eng-001    role=DEVELOPER    grade=EXPERT         score=0.8822
    capability_text: Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Docker

정보처리기사
  engineer_id=eng-013    role=DEVELOPER    grade=SENIOR         score=0.8822
    capability_text: Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Docker

정보처리기사
  engineer_id=eng-025    role=DEVELOPER    grade=SENIOR         score=0.8822
    capability_text: Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Docker

정보처리기사


In [34]:
# 5-4. DualUpstashStore 검증 — experience 검색
exp_query = "[프로젝트] 현대차 ERP 시스템 개발: 제조업 ERP 시스템 재구축\n[포지션] 백엔드 개발자"
print(f"=== experience 검색 top-3 ===")
exp_results = dual_store.search_experience(exp_query, top_k=3)
for r in exp_results:
    eid  = r.metadata.get("engineer_id", "?")
    role = r.metadata.get("engineer_role", "?")
    grade = r.metadata.get("grade", "?")
    print(f"  engineer_id={eid:<10s} role={role:<12s} grade={grade:<14s} score={r.score:.4f}")

=== experience 검색 top-3 ===
  engineer_id=eng-001    role=DEVELOPER    grade=EXPERT         score=0.8766
  engineer_id=eng-037    role=DEVELOPER    grade=INTERMEDIATE   score=0.8746
  engineer_id=eng-033    role=DEVELOPER    grade=INTERMEDIATE   score=0.8738


In [35]:
# 5-5. 카테고리별 스킬 분포 확인
from collections import Counter

category_counter = Counter(s.category for s in SAMPLE_SKILLS)
print("=== SAMPLE_SKILLS 카테고리별 분포 ===")
for cat, cnt in sorted(category_counter.items()):
    print(f"  {cat:<12s}: {cnt}개")
print(f"  {'TOTAL':<12s}: {len(SAMPLE_SKILLS)}개")

=== SAMPLE_SKILLS 카테고리별 분포 ===
  CLOUD       : 10개
  DATABASE    : 10개
  DESIGN      : 8개
  DEVOPS      : 10개
  FRAMEWORK   : 12개
  INFRA       : 8개
  LANGUAGE    : 10개
  OTHER       : 8개
  TOOL        : 8개
  TOTAL       : 84개


## 완료

```
DualUpstashStore
  capability  namespace : SAMPLE_ENGINEER_PROFILES capability_text 100건
  experience  namespace : SAMPLE_ENGINEER_PROFILES experience_text 100건

CapabilityMasterStore
  capability_master namespace : SAMPLE_SKILLS 84건
```

이후 노트북에서 위 데이터를 검색만 해서 재활용한다. 재임베딩 없음.